In [22]:
import pandas as pd
import joblib
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent
from langchain_experimental.plan_and_execute import PlanAndExecute, load_chat_planner, load_agent_executor
from langchain.schema import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import ChatOpenAI
from sklearn.model_selection import train_test_split
import os
import json
import time

In [2]:
# Load data
data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')
data.head()

/var/folders/s1/z7p70y8n35lcdk9lq0wvfxz00000gq/T/ipykernel_62646/1113152288.py:2: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,50 to 69,107,F,White,Not Span/Hispanic,...,Major,Major,Medical,Medicaid,NaN,NaN,NaN,Y,"51,514.62","7,552.54"
1,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,M,Black/African American,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"25,370.86","3,469.55"
2,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,F,Other Race,Spanish/Hispanic,...,Minor,Minor,Medical,Medicaid,NaN,NaN,NaN,N,"23,876.78","6,180.33"
3,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,100,F,Black/African American,Not Span/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"43,319.05","12,588.93"
4,New York City,Bronx,7000006.0,1168.0,Montefiore Medical Center-Wakefield Hospital,18 to 29,104,M,Other Race,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,"40,266.23","10,355.99"


In [3]:
import json

with open("../data/data_info.json", "r") as f:
    data_info = json.load(f)

In [4]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

Hospital Service Area 5390
Hospital County 5390
Operating Certificate Number 5961
Permanent Facility Id 5390
Zip Code - 3 digits 41227
CCSR Procedure Code 582815
CCSR Procedure Description 582815
APR Severity of Illness Description 636
APR Risk of Mortality 636
Payment Typology 2 1121497
Payment Typology 3 1814362
Birth Weight 1894796


In [5]:
# Handle missing data and changing column type
for col_name, info in data_info.items():
    print(col_name)
    if info["type"] == "int64":
        if col_name == "Zip Code - 3 digits":
            data[col_name] = data[col_name].replace("OOS", 000)
            data[col_name] = data[col_name].fillna(000)
        if col_name == "Length of Stay":
            data[col_name] = data[col_name].replace("120 +", 121)
        if col_name == "Birth Weight":
            data[col_name] = data[col_name].replace("UNKN", -1)
            data[col_name] = data[col_name].fillna(-1)
        else:
            data[col_name] = data[col_name].fillna(-1)
            
        data[col_name] = pd.to_numeric(data[col_name]).astype('int')
        
    elif info["type"] == "str":
        data[col_name] = data[col_name].fillna("Unknown")
        data[col_name] = data[col_name].astype(str)

    elif info["type"] == "float64":
        data[col_name] = data[col_name].str.replace(',', '')
        data[col_name] = pd.to_numeric(data[col_name]).astype('float')


Hospital Service Area
Hospital County
Operating Certificate Number
Permanent Facility Id
Facility Name
Age Group
Zip Code - 3 digits
Gender
Race
Ethnicity
Length of Stay
Type of Admission
Patient Disposition
Discharge Year
CCSR Diagnosis Code
CCSR Diagnosis Description
CCSR Procedure Code
CCSR Procedure Description
APR DRG Code
APR DRG Description
APR MDC Code
APR MDC Description
APR Severity of Illness Code
APR Severity of Illness Description
APR Risk of Mortality
APR Medical Surgical Description
Payment Typology 1
Payment Typology 2
Payment Typology 3
Birth Weight
Emergency Department Indicator
Total Charges
Total Costs


In [6]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

In [7]:
for col in data.columns:
    print(col, data[col].dtype)

Hospital Service Area object
Hospital County object
Operating Certificate Number int64
Permanent Facility Id int64
Facility Name object
Age Group object
Zip Code - 3 digits int64
Gender object
Race object
Ethnicity object
Length of Stay int64
Type of Admission object
Patient Disposition object
Discharge Year int64
CCSR Diagnosis Code object
CCSR Diagnosis Description object
CCSR Procedure Code object
CCSR Procedure Description object
APR DRG Code int64
APR DRG Description object
APR MDC Code int64
APR MDC Description object
APR Severity of Illness Code object
APR Severity of Illness Description object
APR Risk of Mortality object
APR Medical Surgical Description object
Payment Typology 1 object
Payment Typology 2 object
Payment Typology 3 object
Birth Weight int64
Emergency Department Indicator object
Total Charges float64
Total Costs float64


In [8]:
# Identify categorical columns and convert them to categories
categorical_columns = []
for col in data.columns:
    if data[col].dtype == "object":
        data[col] = data[col].astype('category')

In [9]:
X = data.drop(columns=['Total Costs', ])
y = data['Total Costs']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
model = joblib.load("../models/xgb_model_best.pkl")

In [12]:
# load shap explainer
explainer = joblib.load('../models/shap_explainer_xgb_best.pkl')

/Users/mahsaamani/Downloads/Saarlanduni/DataScience/UnveilingHospitalCostDrivers/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
SYSTEM_PROMPT = '''
You are an expert in healthcare cost optimization and hospital operations strategy.

Your role is to analyze structured data from individual inpatient hospital cases. Each feature includes:
- The data type (e.g., int, float, str)
- The observed value for the case
- The SHAP value, which quantifies how much that feature contributed to the total inpatient cost
- A list of possible values the feature can take (if available)

Your goal is to:
1. Identify which features are the most significant cost drivers.
2. Propose feasible, actionable strategies to reduce their cost impact, based on the available options.
3. Ensure your recommendations do not compromise—ideally improve—the quality of care.

Respond only with specific, pragmatic suggestions tailored to the data provided.
'''

USER_PROMPT = f'''
You are given structured data for a single inpatient case.

Each feature contains:
- "type": the data type
- "value": the observed value
- "shap value": contribution to the total inpatient cost
- "other options": the possible values for that feature (if any)

Your tasks:
1. For each feature, suggest a specific, feasible strategy to reduce its cost impact, using the "other options" where applicable to that patient. If there is no strategy, leave this empty.
2. Conclude with a short summary (2–3 sentences) explaining the overall cost-reduction logic you applied.
3. Return your response in this JSON format only (no additional text):

{{
  "Feature Name 1": "strategy",
  "Feature Name 2": "strategy",
  ...
  "Summary": "your overall summary"
}}

Data:
###patient_info###
'''

# Sample input

In [14]:
patient_dict = {
    "Hospital Service Area": "New York City",
    "Hospital County": "Kings",
    "Operating Certificate Number": 7001009,
    "Permanent Facility Id": 1294,
    "Facility Name": "Coney Island Hospital",
    "Age Group": "18 to 29",
    "Zip Code - 3 digits": 112,
    "Gender": "F",
    "Race": "Black/African American",
    "Ethnicity": "Not Span/Hispanic",
    "Length of Stay": 1,
    "Type of Admission": "Emergency",
    "Patient Disposition": "Home or Self Care",
    "Discharge Year": 2022,
    "CCSR Diagnosis Code": "INF012",
    "CCSR Diagnosis Description": "COVID-19",
    "CCSR Procedure Code": "ADM015",
    "CCSR Procedure Description": "ADMINISTRATION OF ANTIBIOTICS",
    "APR DRG Code": 137,
    "APR DRG Description": "MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS",
    "APR MDC Code": 4,
    "APR MDC Description": "DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM",
    "APR Severity of Illness Code": 3,
    "APR Severity of Illness Description": "Major",
    "APR Risk of Mortality": "Moderate",
    "APR Medical Surgical Description": "Medical",
    "Payment Typology 1": "Medicaid",
    "Payment Typology 2": "Unknown",
    "Payment Typology 3": "Unknown",
    "Birth Weight": -1,
    "Emergency Department Indicator": "Y",
    "Total Charges": 8803.14
}

In [15]:
shared_memory = {}

def extract_shap_info():
    patient_df = pd.DataFrame(patient_dict, index=[0])
    for col in data.select_dtypes(['category']).columns:
        patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

    shared_memory["current cost"] = model.predict(patient_df).item()
    shap_values = explainer(patient_df)
    shap_info = {}
    for i, col in enumerate(patient_df.columns):
        shap_info[col] = {
            "type": data_info[col]["type"],
            "value": shap_values.data[0][i],
            "shap value": float(shap_values.values[0][i]),
            "other options": data_info[col].get("options", [])
        }
    return json.dumps(shap_info)


def suggest_strategies(shap_info: str):
    shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
    response = llm.invoke(
    [
        HumanMessage(
            content=USER_PROMPT.replace(
                "###patient_info###", json.dumps(shap_info)
            )
        ),
        SystemMessage(content=SYSTEM_PROMPT),
    ]
    )
    strategies = response.content.strip()
    return json.dumps(strategies)


def cost_predition(strategies: dict):
    strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
    shared_memory["suggested strategies"] = strategies
    min_cost = shared_memory["current cost"]
    for col, method in strategies.items():
        if col == "Summary":
            continue
        info = data_info[col]
        if method != None and len(method) > 5:
            if info["type"] == "str" and "options" in info:
                # print(col, info["options"])
                for option in info["options"]:
                    updated_patient_dict = patient_dict.copy()
                    # apply changes
                    updated_patient_dict[col] = option
                    updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                    for col_n in data.select_dtypes(['category']).columns:
                        updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                    cost = model.predict(updated_patient_df).item()
                    if cost < min_cost:
                        min_cost = cost
                        shared_memory["new cost"] = min_cost
                        shared_memory["target feature"] = col
                        shared_memory["target strategy"] = option
    print(shared_memory)
    return json.dumps(shared_memory)

            
# --- LLM Setup ---
gemini_api_key = os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

# --- Tool Collection and Agent Setup ---
tools = [
    StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
    StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
    StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
]


planner = load_chat_planner(llm)
executor = load_agent_executor(llm=llm, tools=tools, verbose=True)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=True, input_key="input")

# --- Execute Instruction ---
if __name__ == "__main__":
    prompt = f"""
        You are an expert in healthcare cost optimization and hospital operations strategy.

        Execute these steps using the available tools:
        
        1. Run ExtractSHAPInfo with the result.
        2. Run SuggestStrategies with the result.
        3. Run CostPredition with suggested strategies.
        Return the final recommended strategies.
        """
    
    max_retries = 5
    for attempt in range(max_retries):
        try:
            output = agent.invoke({"input": prompt})
            # json_output = json.loads(output) 
            print("Success!")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10)
    else:
        print("All retries failed.")




> Entering new PlanAndExecute chain...
steps=[Step(value='Run ExtractSHAPInfo.'), Step(value='Run SuggestStrategies with the output of ExtractSHAPInfo.'), Step(value='Run CostPrediction with the strategies suggested by SuggestStrategies.'), Step(value='Report the final recommended strategies from the CostPrediction output.')]

> Entering new AgentExecutor chain...
Action:
```
{
  "action": "ExtractSHAPInfo",
  "action_input": {}
}
```
Observation: {"Hospital Service Area": {"type": "str", "value": "New York City", "shap value": -23.826786041259766, "other options": ["Capital/Adirondack", "Central NY", "Finger Lakes", "Hudson Valley", "Long Island", "New York City", "Southern Tier", "Western NY"]}, "Hospital County": {"type": "str", "value": "Kings", "shap value": 1.8218145370483398, "other options": ["Albany", "Allegany", "Bronx", "Broome", "Cattaraugus", "Cayuga", "Chautauqua", "Chemung", "Chenango", "Clinton", "Columbia", "Cortland", "Delaware", "Dutchess", "Erie", "Essex", "Frankl

In [16]:
shared_memory, output["output"]

({'current cost': 3626.18017578125,
  'suggested strategies': {'Hospital Service Area': "Transferring the patient to a hospital in a lower-cost service area like 'Capital/Adirondack'",
   'Facility Name': "Transferring the patient to a lower-cost facility within the same service area, like 'Redacted for Confidentiality'",
   'APR Severity of Illness Code': "Ensuring accurate coding to reflect the true severity of illness, potentially coding as 'Moderate' or lower"},
  'new cost': -971.3054809570312,
  'target feature': 'Facility Name',
  'target strategy': 'Montefiore Medical Center - Henry & Lucy Moses Div'},
 "The recommended cost-reducing strategies are:\n\n*   **Hospital Service Area:** Transferring the patient to a hospital in a lower-cost service area like 'Capital/Adirondack'.\n*   **Facility Name:** Transferring the patient to a lower-cost facility within the same service area, like 'Redacted for Confidentiality'.\n*   **APR Severity of Illness Code:** Ensuring accurate coding 

# Evaluation

In [32]:
for pindex in range(10):
    print(f"_________Sample {pindex}___________")
    patient_dict = X_test.iloc[pindex].to_dict()
    results = {"patient_dict": patient_dict}
    results["attempt"] = -1
    shared_memory = {}

    def extract_shap_info():
        patient_df = pd.DataFrame(patient_dict, index=[0])
        for col in data.select_dtypes(['category']).columns:
            patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

        shared_memory["current cost"] = model.predict(patient_df).item()
        shap_values = explainer(patient_df)
        shap_info = {}
        for i, col in enumerate(patient_df.columns):
            shap_info[col] = {
                "type": data_info[col]["type"],
                "value": shap_values.data[0][i],
                "shap value": float(shap_values.values[0][i]),
                "other options": data_info[col].get("options", [])
            }
        return json.dumps(shap_info)


    def suggest_strategies(shap_info: str):
        shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
        response = llm.invoke(
        [
            HumanMessage(
                content=USER_PROMPT.replace(
                    "###patient_info###", json.dumps(shap_info)
                )
            ),
            SystemMessage(content=SYSTEM_PROMPT),
        ]
        )
        strategies = response.content.strip()
        return json.dumps(strategies)


    def cost_predition(strategies: dict):
        strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
        shared_memory["suggested strategies"] = strategies
        min_cost = shared_memory["current cost"]
        for col, method in strategies.items():
            if col == "Summary":
                continue
            info = data_info[col]
            if method != None and len(method) > 5:
                if info["type"] == "str" and "options" in info:
                    # print(col, info["options"])
                    for option in info["options"]:
                        updated_patient_dict = patient_dict.copy()
                        # apply changes
                        updated_patient_dict[col] = option
                        updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                        for col_n in data.select_dtypes(['category']).columns:
                            updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                        cost = model.predict(updated_patient_df).item()
                        if cost < min_cost:
                            min_cost = cost
                            shared_memory["new cost"] = min_cost
                            shared_memory["target feature"] = col
                            shared_memory["target strategy"] = option
        # print(shared_memory)
        return json.dumps(shared_memory)

    # --- LLM Setup ---
    model_name = "gemini-2.0-flash"

    gemini_api_key = os.getenv("GEMINI_API_KEY")
    llm = ChatGoogleGenerativeAI(model=model_name, api_key=gemini_api_key)

    # or_api_key = os.getenv("OR_API_KEY")
    # llm = ChatOpenAI(base_url="https://openrouter.ai/api/v1", api_key=gemini_api_key, model=model_name)


    tools = [
        StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
        StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
        StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
    ]

    planner = load_chat_planner(llm)
    executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
    agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")


    prompt = f"""
    You are an expert in healthcare cost optimization and hospital operations strategy.

    Execute these steps using the available tools:

    1. Run ExtractSHAPInfo with the result.
    2. Run SuggestStrategies with the result.
    3. Run CostPredition with suggested strategies.
    Return the final recommended strategies.
    """
    max_retries = 5

    def write_in_json(i, data):
        os.makedirs(f"../results/{model_name}/", exist_ok=True)
        with open(f"../results/{model_name}/{i}.json", "w", encoding='utf8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    
    start = time.time()
    for attempt in range(max_retries):
        try:
            output = agent.invoke({"input": prompt})
            print("Success!")
            results["attempt"] = attempt + 1
            results["llm_response"] = output["output"]
            # print(shared_memory, output["output"])
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}.")
            time.sleep(10)
    else:
        print("All retries failed.")
        time.sleep(10)
    end = time.time()

    results["response time"] = end - start
    results.update(shared_memory)
    print(results)
    write_in_json(pindex, results)


_________Sample 0___________
Success!
{'patient_dict': {'Hospital Service Area': 'Western NY', 'Hospital County': 'Erie', 'Operating Certificate Number': 1401014, 'Permanent Facility Id': 207, 'Facility Name': 'Buffalo General Medical Center', 'Age Group': '70 or Older', 'Zip Code - 3 digits': 140, 'Gender': 'F', 'Race': 'White', 'Ethnicity': 'Not Span/Hispanic', 'Length of Stay': 3, 'Type of Admission': 'Emergency', 'Patient Disposition': 'Home or Self Care', 'Discharge Year': 2022, 'CCSR Diagnosis Code': 'INJ008', 'CCSR Diagnosis Description': 'Traumatic brain injury (TBI); concussion, initial encounter', 'CCSR Procedure Code': 'Unknown', 'CCSR Procedure Description': 'Unknown', 'APR DRG Code': 55, 'APR DRG Description': 'HEAD TRAUMA WITH COMA > 1 HOUR OR HEMORRHAGE', 'APR MDC Code': 1, 'APR MDC Description': 'DISEASES AND DISORDERS OF THE NERVOUS SYSTEM', 'APR Severity of Illness Code': '1', 'APR Severity of Illness Description': 'Minor', 'APR Risk of Mortality': 'Moderate', 'APR Me